<a href="https://colab.research.google.com/github/Qophy/PBML/blob/DataSet/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
import os
import sys
import tensorflow as tf
import pandas as pd
import numpy as np
#import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns


In [3]:
os.listdir('/content/drive/My Drive/code')

['INWSStaticDataGroupedByTimestamp1.xlsx',
 'pinns_tf2-main',
 'PINN-PDE(utube)',
 'PINNs-master(I)']

In [4]:
# Define the path to your modules directory on Google Drive
# Adjust 'My Drive/your_project_folder' if your modules are in a different location
data_path = '/content/drive/My Drive/code/'

# Add the modules_path to sys.path if it's not already there
if data_path not in sys.path:
    sys.path.append(data_path)

In [5]:
os.listdir(data_path)

['INWSStaticDataGroupedByTimestamp1.xlsx',
 'pinns_tf2-main',
 'PINN-PDE(utube)',
 'PINNs-master(I)']

In [6]:
dataset = pd.read_excel(data_path + 'INWSStaticDataGroupedByTimestamp1.xlsx')

In [39]:
dataset.sample(10)

,Id,Node Id,Timestamp,Timestamp as DateTime,Temperature,Conductivity,pH,DissolvedOxygen
23080,23081,2,201505021507070016,2015/05/02 15:07:07 0000,12.49,307.5,5.27,64.9
3835,3836,1,201507142143390016,2015/07/14 21:43:39 0000,22.21,395.2,7.97,9.3
10775,10776,1,201509220123500000,2015/09/22 01:23:50 0000,14.40,350.5,6.30,15.1
26174,26175,2,201506191819100000,2015/06/19 18:19:10 0000,18.20,386.7,6.51,1.4
5233,5234,1,201507242339100000,2015/07/24 23:39:10 0000,24.18,428.0,6.27,7.4
24193,24194,2,201506051648070016,2015/06/05 16:48:07 0000,18.89,325.9,6.24,32.8
19296,19297,1,201512230358327008,2015-12-23 03:58:32,NaN,NaN,6.27,NaN
13410,13411,1,201510190113520000,2015/10/19 01:13:52 0000,14.35,297.1,2.59,19.8
22737,22738,1,201601051823161984,2016-01-05 18:23:16,NaN,NaN,6.14,NaN
21962,21963,1,201601010424369984,2016/01/01 04:24:37 0000,2.82,263.0,6.21,8.5


## Find and Drop NaNs in the DATASET.

In [8]:
def print_rows_with_nans(df):
    nan_rows = df[df.isna().any(axis=1)]
    if nan_rows.empty:
        print(f"\n✅ No NaN values found in the dataset.")
    else:
        print(f"\n⚠️ Found {len(nan_rows)} rows with NaN values.")
        # print((nan_rows))

    return nan_rows

In [40]:
print_rows_with_nans(dataset)


⚠️ Found 4642 rows with NaN values.


,Id,Node Id,Timestamp,Timestamp as DateTime,Temperature,Conductivity,pH,DissolvedOxygen
0,1,1,201505011710380000,2015/05/01 17:10:38 0000,15.12,345.3,NaN,NaN
220,221,1,201505161519360000,2015/05/16 15:19:36 0000,16.06,370.1,NaN,69.5
376,377,1,201505291858129984,2015/05/29 18:58:13 0000,NaN,324.9,7.76,6.2
377,378,1,201505291902060000,2015/05/29 19:02:06 0000,14.74,325.0,NaN,5.7
3752,3753,1,201507140738500000,2015/07/14 07:38:50 0000,20.97,562.8,NaN,5.6
...,...,...,...,...,...,...,...,...
23069,23070,1,201601052108076992,2016-01-05 21:08:07,NaN,NaN,6.26,NaN
23070,23071,1,201601052108100000,2016/01/05 21:08:10 0000,NaN,NaN,NaN,9.4
23071,23072,1,201601052108100992,2016-01-05 21:08:10,NaN,257.2,NaN,NaN
23072,23073,1,201601052108124000,2016-01-05 21:08:12,2.59,NaN,NaN,NaN


In [41]:
nan_row_indices = print_rows_with_nans(dataset).index.tolist()


⚠️ Found 4642 rows with NaN values.


In [42]:
def drop_nan_rows(list_indices):
  dataset_shape = dataset.shape
  dataset_cleaned = dataset.drop(nan_row_indices)
  dataset_cleaned_shape = dataset_cleaned.shape

  return [dataset_cleaned, dataset_cleaned_shape, dataset_shape]

In [43]:
wq_data = drop_nan_rows(nan_row_indices)[0].drop(columns=["DissolvedOxygen", "Timestamp"])
print(wq_data.sample(5))

          Id  Node Id     Timestamp as DateTime  Temperature  Conductivity  \
15773  15774        1  2015/11/13 19:35:21 0000         9.73         305.4   
16138  16139        1  2015/11/18 02:55:01 0000         9.38         262.0   
23354  23355        2  2015/05/22 19:33:15 0000        15.79         306.3   
6535    6536        1  2015/08/16 16:34:48 0000        19.90         296.3   
9147    9148        1  2015/09/08 20:58:20 0000        14.55         298.7   

         pH  
15773  7.58  
16138  7.91  
23354  8.64  
6535   5.52  
9147   4.27  


## Cascading wq_data into two different measuring points

In [44]:
wq_data_node1 = wq_data[wq_data["Node Id"] == 1].copy()
wq_data_node2 = wq_data[wq_data["Node Id"] == 2].copy()

In [45]:
wq_data_node1.sample(5)


,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH
13455,13456,1,2015/10/22 15:48:16 0000,9.95,309.9,2.05
8868,8869,1,2015/09/06 14:37:03 0000,17.01,315.8,2.25
10740,10741,1,2015/09/21 18:57:03 0000,13.32,289.8,6.25
5957,5958,1,2015/08/12 14:01:02 0000,18.53,283.6,5.19
6340,6341,1,2015/08/15 07:19:48 0000,18.65,313.8,6.60


In [46]:
wq_data_node2.sample(5)

,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH
24749,24750,2,2015/06/09 15:07:26 0000,17.87,326.3,6.15
25781,25782,2,2015/06/16 22:39:43 0000,21.53,452.1,6.36
27795,27796,2,2015/07/06 01:49:22 0000,20.12,401.4,7.73
26574,26575,2,2015/06/22 14:10:00 0000,81.91,0.9,-18.74
29750,29751,2,2015/08/29 18:41:30 0000,-0.33,-5021.0,-1.00


# Create a FOLDER and name it PINN_WQS_dataset


In [ ]:
#os.makedirs('/content/drive/My Drive/PINN_WQS_dataset', exist_ok=True)

In [ ]:
#os.listdir('/content/drive/My Drive/PINN_WQS_dataset')

[]

# Save files to PINN_WQS_dataset directory

In [ ]:
#wq_data_node1.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_1.csv', index=False)

In [ ]:
#wq_data_node2.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_2.csv', index=False)

# Convert the TimeStamp feature into Cyclic features

In [47]:
wq_data_node1["Time"] = pd.to_datetime(wq_data_node1["Timestamp as DateTime"].str[:-5], format='%Y/%m/%d %H:%M:%S')
wq_data_node2["Time"] = pd.to_datetime(wq_data_node2["Timestamp as DateTime"].str[:-5], format='%Y/%m/%d %H:%M:%S')

In [48]:
#Node 1 Data
wq_data_node1["Hour"] = wq_data_node1["Time"].dt.hour
wq_data_node1["DOW"] = wq_data_node1["Time"].dt.dayofweek
wq_data_node1["Mon"] = wq_data_node1["Time"].dt.month

In [55]:
wq_data_node2["Hour"] = wq_data_node2["Time"].dt.hour
wq_data_node2["DOW"] = wq_data_node2["Time"].dt.dayofweek
wq_data_node2["Mon"] = wq_data_node2["Time"].dt.month

In [54]:
wq_data_node1.head()

,Temperature,Conductivity,pH,Time,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
1,15.22,345.7,7.36,2015-05-01 17:20:48,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
2,15.80,343.9,7.56,2015-05-01 17:30:59,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
3,16.25,339.7,7.00,2015-05-01 17:41:10,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
4,-0.33,342.7,7.35,2015-05-01 17:51:20,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
5,12.67,328.4,6.10,2015-05-02 14:36:17,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


In [56]:
wq_data_node2.head()

,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH,Time,Hour,DOW,Mon
23073,23074,2,2015/05/01 17:17:54 0000,14.96,337.4,6.35,2015-05-01 17:17:54,17,4,5
23074,23075,2,2015/05/01 17:28:05 0000,14.98,337.9,6.77,2015-05-01 17:28:05,17,4,5
23075,23076,2,2015/05/01 17:38:16 0000,15.04,340.3,6.57,2015-05-01 17:38:16,17,4,5
23076,23077,2,2015/05/01 17:48:26 0000,15.10,341.4,6.46,2015-05-01 17:48:26,17,4,5
23077,23078,2,2015/05/02 14:36:34 0000,12.46,297.1,6.30,2015-05-02 14:36:34,14,5,5


In [50]:
wq_data_node1["hour_sin"] =  np.sin(2 * np.pi * wq_data_node1['Hour'] / 24)
wq_data_node1["hour_cos"] =  np.cos(2 * np.pi * wq_data_node1['Hour'] / 24)

wq_data_node1["DOW_sin"] =  np.sin(2 * np.pi * wq_data_node1['DOW'] / 7)
wq_data_node1["DOW_cos"] =  np.cos(2 * np.pi * wq_data_node1['DOW'] / 7)

wq_data_node1["Mon_sin"] =  np.sin(2 * np.pi * wq_data_node1['Mon'] / 12)
wq_data_node1["Mon_cos"] =  np.cos(2 * np.pi * wq_data_node1['Mon'] / 12)

In [57]:
wq_data_node2["hour_sin"] =  np.sin(2 * np.pi * wq_data_node2['Hour'] / 24)
wq_data_node2["hour_cos"] =  np.cos(2 * np.pi * wq_data_node2['Hour'] / 24)

wq_data_node2["DOW_sin"] =  np.sin(2 * np.pi * wq_data_node2['DOW'] / 7)
wq_data_node2["DOW_cos"] =  np.cos(2 * np.pi * wq_data_node2['DOW'] / 7)

wq_data_node2["Mon_sin"] =  np.sin(2 * np.pi * wq_data_node2['Mon'] / 12)
wq_data_node2["Mon_cos"] =  np.cos(2 * np.pi * wq_data_node2['Mon'] / 12)

In [58]:
wq_data_node1.head(5)

,Temperature,Conductivity,pH,Time,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
1,15.22,345.7,7.36,2015-05-01 17:20:48,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
2,15.80,343.9,7.56,2015-05-01 17:30:59,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
3,16.25,339.7,7.00,2015-05-01 17:41:10,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
4,-0.33,342.7,7.35,2015-05-01 17:51:20,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
5,12.67,328.4,6.10,2015-05-02 14:36:17,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


In [59]:
wq_data_node2.head(5)

,Id,Node Id,Timestamp as DateTime,Temperature,Conductivity,pH,Time,Hour,DOW,Mon,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
23073,23074,2,2015/05/01 17:17:54 0000,14.96,337.4,6.35,2015-05-01 17:17:54,17,4,5,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23074,23075,2,2015/05/01 17:28:05 0000,14.98,337.9,6.77,2015-05-01 17:28:05,17,4,5,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23075,23076,2,2015/05/01 17:38:16 0000,15.04,340.3,6.57,2015-05-01 17:38:16,17,4,5,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23076,23077,2,2015/05/01 17:48:26 0000,15.10,341.4,6.46,2015-05-01 17:48:26,17,4,5,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23077,23078,2,2015/05/02 14:36:34 0000,12.46,297.1,6.30,2015-05-02 14:36:34,14,5,5,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


In [52]:
wq_data_node1.drop(columns=["Id", "Node Id", "Timestamp as DateTime", "Hour", "DOW", "Mon"], inplace=True)

In [60]:
wq_data_node2.drop(columns=["Id", "Node Id", "Timestamp as DateTime", "Hour", "DOW", "Mon"], inplace=True)

In [61]:
wq_data_node1.head()

,Temperature,Conductivity,pH,Time,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
1,15.22,345.7,7.36,2015-05-01 17:20:48,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
2,15.80,343.9,7.56,2015-05-01 17:30:59,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
3,16.25,339.7,7.00,2015-05-01 17:41:10,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
4,-0.33,342.7,7.35,2015-05-01 17:51:20,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
5,12.67,328.4,6.10,2015-05-02 14:36:17,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


In [62]:
wq_data_node2.head()

,Temperature,Conductivity,pH,Time,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
23073,14.96,337.4,6.35,2015-05-01 17:17:54,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23074,14.98,337.9,6.77,2015-05-01 17:28:05,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23075,15.04,340.3,6.57,2015-05-01 17:38:16,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23076,15.10,341.4,6.46,2015-05-01 17:48:26,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23077,12.46,297.1,6.30,2015-05-02 14:36:34,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


# Save files with DateTime and Cycles to PINN_WQS_dataset directory

In [63]:
wq_data_node1.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_datetime_1.csv', index=False)

In [64]:
wq_data_node2.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_datetime2.csv', index=False)

# Save files with DateTime and Cycles to PINN_WQS_dataset directory

In [65]:
wq_data_node1.drop(columns=["Time"], inplace=True)

In [66]:
wq_data_node2.drop(columns=["Time"], inplace=True)

In [67]:
wq_data_node1.head()
wq_data_node2.head()

,Temperature,Conductivity,pH,hour_sin,hour_cos,DOW_sin,DOW_cos,Mon_sin,Mon_cos
23073,14.96,337.4,6.35,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23074,14.98,337.9,6.77,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23075,15.04,340.3,6.57,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23076,15.10,341.4,6.46,-0.965926,-0.258819,-0.433884,-0.900969,0.5,-0.866025
23077,12.46,297.1,6.30,-0.500000,-0.866025,-0.974928,-0.222521,0.5,-0.866025


In [68]:
wq_data_node1.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_cycles_1.csv', index=False)

In [69]:
wq_data_node2.to_csv('/content/drive/My Drive/PINN_WQS_dataset/wq_dataset_node_cycles_2.csv', index=False)